In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras

In [3]:
df = pd.read_csv('/content/sample_data/classData.csv')
df.head()

,G,C,B,A,Ia,Ib,Ic,Va,Vb,Vc
0,1,0,0,1,-151.291812,-9.677452,85.800162,0.400750,-0.132935,-0.267815
1,1,0,0,1,-336.186183,-76.283262,18.328897,0.312732,-0.123633,-0.189099
2,1,0,0,1,-502.891583,-174.648023,-80.924663,0.265728,-0.114301,-0.151428
3,1,0,0,1,-593.941905,-217.703359,-124.891924,0.235511,-0.104940,-0.130570
4,1,0,0,1,-643.663617,-224.159427,-132.282815,0.209537,-0.095554,-0.113983


In [4]:
df['Combined_G_C_B_A'] = df['G'].astype(str) + df['C'].astype(str) + df['B'].astype(str) + df['A'].astype(str)
df = df.drop(columns=['G', 'C', 'B', 'A'])
display(df.head())

,Ia,Ib,Ic,Va,Vb,Vc,Combined_G_C_B_A
0,-151.291812,-9.677452,85.800162,0.400750,-0.132935,-0.267815,1001
1,-336.186183,-76.283262,18.328897,0.312732,-0.123633,-0.189099,1001
2,-502.891583,-174.648023,-80.924663,0.265728,-0.114301,-0.151428,1001
3,-593.941905,-217.703359,-124.891924,0.235511,-0.104940,-0.130570,1001
4,-643.663617,-224.159427,-132.282815,0.209537,-0.095554,-0.113983,1001


In [49]:
df['Combined_G_C_B_A_mapped'] = pd.factorize(df['Combined_G_C_B_A'])[0] + 1
display(df[['Combined_G_C_B_A', 'Combined_G_C_B_A_mapped']].head())

KeyError: 'Combined_G_C_B_A'

In [8]:
df = df.drop(columns=['Combined_G_C_B_A'])
df = df.rename(columns={'Combined_G_C_B_A_mapped': 'target'})
display(df.head())

,Ia,Ib,Ic,Va,Vb,Vc,target
0,-151.291812,-9.677452,85.800162,0.400750,-0.132935,-0.267815,1
1,-336.186183,-76.283262,18.328897,0.312732,-0.123633,-0.189099,1
2,-502.891583,-174.648023,-80.924663,0.265728,-0.114301,-0.151428,1
3,-593.941905,-217.703359,-124.891924,0.235511,-0.104940,-0.130570,1
4,-643.663617,-224.159427,-132.282815,0.209537,-0.095554,-0.113983,1


In [9]:
df['target'].value_counts()

,count
target,
6,2365
2,1134
5,1133
1,1129
4,1096
3,1004


In [10]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df.drop('target', axis=1)
y = df['target']

# Split the data into training and testing sets
# test_size=0.2 means 20% of the data will be used for testing
# random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (6288, 6)
X_test shape: (1573, 6)
y_train shape: (6288,)
y_test shape: (1573,)


In [11]:
from sklearn.preprocessing import MinMaxScaler

# Create a MinMaxScaler object
scaler = MinMaxScaler()

# Fit the scaler to the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Convert the scaled array back to a DataFrame for easier viewing, maintaining column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)

display(X_train_scaled.head())

,Ia,Ib,Ic,Va,Vb,Vc
0,0.004464,0.471883,0.495997,0.581081,0.628175,0.295585
1,0.525324,0.489753,0.480792,0.845155,0.643532,0.015167
2,0.506706,0.796374,0.194813,0.888479,0.515703,0.101974
3,0.463784,0.871726,0.161668,0.403707,0.471258,0.633311
4,0.908633,0.528971,0.508073,0.550884,0.187351,0.775022


In [12]:
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical

# Convert y_train and y_test to 0-indexed for to_categorical
y_train_0_indexed = y_train - 1
y_test_0_indexed = y_test - 1

# Determine the number of classes (output units)
num_classes = y_train_0_indexed.nunique()

# One-hot encode the target variable
y_train_categorical = to_categorical(y_train_0_indexed, num_classes=num_classes)
y_test_categorical = to_categorical(y_test_0_indexed, num_classes=num_classes)

print(f"Original y_train shape: {y_train.shape}")
print(f"One-hot encoded y_train shape: {y_train_categorical.shape}")

Original y_train shape: (6288,)
One-hot encoded y_train shape: (6288, 6)


In [44]:
# Define the input dimension
input_dim = X_train_scaled.shape[1]

# Initialize the Sequential model
model = Sequential()

# Input layer and first hidden layer
model.add(Dense(64, activation='relu', input_shape=(input_dim,)))

# Second hidden layer
model.add(Dense(16, activation='relu'))

# Third hidden layer
#model.add(Dense(32, activation='relu'))

# fourth hidden layer
#model.add(Dense(16, activation='relu'))

# Output layer (num_classes units with softmax for multi-class classification)
model.add(Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_25 (Dense)                │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 6)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,590 (6.21 KB)

 Trainable params: 1,590 (6.21 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
history = model.fit(X_train_scaled, y_train_categorical, epochs=50, batch_size=16, validation_split=0.2, verbose=1)

print("\nTraining History:")
print(history.history)

Epoch 1/50
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3179 - loss: 1.6408 - val_accuracy: 0.4555 - val_loss: 1.4302
Epoch 2/50
315/315 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5187 - loss: 1.1945 - val_accuracy: 0.6089 - val_loss: 0.9816
Epoch 3/50
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6485 - loss: 0.8719 - val_accuracy: 0.6908 - val_loss: 0.7546
Epoch 4/50
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7121 - loss: 0.7084 - val_accuracy: 0.7075 - val_loss: 0.6547
Epoch 5/50
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7374 - loss: 0.6159 - val_accuracy: 0.7583 - val_loss: 0.5693
Epoch 6/50
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7543 - loss: 0.5564 - val_accuracy: 0.7655 - val_loss: 0.5236
Epoch 7/50
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7620 - loss: 0.5163 - val_accuracy: 0.7663 - val_loss: 0.4803
Epoch 8/50
315/315 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7797 - loss: 0.4822 - val_accuracy: 0.

### Model Evaluation

In [46]:
# Scale the test data using the *fitted* scaler
X_test_scaled = scaler.transform(X_test)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

display(X_test_scaled.head())

,Ia,Ib,Ic,Va,Vb,Vc
0,0.475969,0.504198,0.515213,0.858905,0.036497,0.619895
1,0.901609,0.542475,0.519309,0.541509,0.187791,0.783974
2,0.888680,0.542421,0.478424,0.279632,0.644098,0.581605
3,0.198867,0.319029,0.465444,0.476137,0.787916,0.238043
4,0.530579,0.500921,0.465337,0.686568,0.809863,0.004695


In [47]:
from sklearn.metrics import classification_report
import numpy as np

# Predict probabilities for the training set
y_train_pred_probs = model.predict(X_train_scaled)
# Convert probabilities to class labels (0-indexed)
y_train_pred_labels = np.argmax(y_train_pred_probs, axis=1)

# Predict probabilities for the test set
y_test_pred_probs = model.predict(X_test_scaled)
# Convert probabilities to class labels (0-indexed)
y_test_pred_labels = np.argmax(y_test_pred_probs, axis=1)

print("\n--- Training Set Classification Report ---")
print(classification_report(y_train_0_indexed, y_train_pred_labels))

print("\n--- Test Set Classification Report ---")
print(classification_report(y_test_0_indexed, y_test_pred_labels))

197/197 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

--- Training Set Classification Report ---
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       887
           1       0.92      0.97      0.94       896
           2       1.00      0.97      0.99       803
           3       0.53      0.47      0.50       897
           4       0.50      0.51      0.51       922
           5       1.00      1.00      1.00      1883

    accuracy                           0.84      6288
   macro avg       0.81      0.82      0.82      6288
weighted avg       0.84      0.84      0.84      6288


--- Test Set Classification Report ---
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       242
           1       0.93      0.95      0.94       238
           2       1.00      0.97      0.98       201
           3       0.50      0.46      0.48       199
           4       0.5

In [41]:
# Find the best epoch based on validation accuracy
best_epoch = np.argmax(history.history['val_accuracy']) + 1 # +1 because epochs are 1-indexed
best_val_accuracy = history.history['val_accuracy'][best_epoch - 1]
best_val_loss = history.history['val_loss'][best_epoch - 1]
best_train_accuracy = history.history['accuracy'][best_epoch - 1]
best_train_loss = history.history['loss'][best_epoch - 1]

print(f"\n--- Best Epoch Summary ---")
print(f"Best Epoch: {best_epoch}")
print(f"Validation Accuracy at Best Epoch: {best_val_accuracy:.4f}")
print(f"Validation Loss at Best Epoch: {best_val_loss:.4f}")
print(f"Training Accuracy at Best Epoch: {best_train_accuracy:.4f}")
print(f"Training Loss at Best Epoch: {best_train_loss:.4f}")


--- Best Epoch Summary ---
Best Epoch: 41
Validation Accuracy at Best Epoch: 0.8506
Validation Loss at Best Epoch: 0.2713
Training Accuracy at Best Epoch: 0.8266
Training Loss at Best Epoch: 0.2848
